# App Demo

Run server in the background using
```bash
pixi run fastapi dev
```
or 
```bash
fastapi dev
```

## Upload Files

In [1]:
import requests

with open("data/pdfs/tenant_alpha/fictional_company_contract.pdf", "rb") as f:
    response_1 = requests.post(
        "http://127.0.0.1:8000/upload/", 
        files={"file": f}, 
        data={"tenant_id": "tenant_alpha", "tenant_token": "tenant_alpha_token"})

with open("data/pdfs/tenant_beta/fictional_company_performance_report.pdf", "rb") as f:
    response_2 = requests.post(
        "http://127.0.0.1:8000/upload/", 
        files={"file": f}, 
        data={"tenant_id": "tenant_beta", "tenant_token": "tenant_beta_token"})

# Sanity Check
response_2.json()


{'file': 'fictional_company_performance_report.pdf',
 'num_chunks': 17,
 'saved_records': 55}

## Query Service

In [2]:
question = "What is company name?"
response = requests.post(
        "http://127.0.0.1:8000/question/", 
        json={
            "question": question, 
            "tenant_id": "tenant_beta", 
            "tenant_token": "tenant_beta_token"
            }
        )
print("Answer for tenant_beta:", response.json())

response = requests.post(
        "http://127.0.0.1:8000/question/", 
        json={
            "question": question, 
            "tenant_id": "tenant_alpha", 
            "tenant_token": "tenant_alpha_token"
            }
        )
print("Answer for tenant_alpha:", response.json())

Answer for tenant_beta: {'question': 'What is company name?', 'answer': 'The company name "Asteron Logistics Group" can be inferred from the provided context, as it appears in the program identifier "ALG-HFP-2026-042".'}
Answer for tenant_alpha: {'question': 'What is company name?', 'answer': 'The company name is Northstar Dynamics Ltd.'}


## RAG is Tenant Isolated

FastAPI server is not required for this test. 

We generate two very similar documents, with key information like company name, founding year, and business sector. All the rest stay similar. 
We are going to show that 

- with the same question, the RAG system will give different answers for the two tenants. 
- when asking a question with keyword from the other tenant, the RAG system will not retrieve context from the other tenant. 

In [1]:
from pdf_rag_poc.embedder import SentenceTransformerEmbedder
from pdf_rag_poc.repository import ChromaDBRepository
from pdf_rag_poc.data_structure import Document

embedder = SentenceTransformerEmbedder()
repository = ChromaDBRepository(embedder)

tenant_alpha_paragraph: str = """Northstar Analytics is a software company founded in 2018. The company produces softwares for business intelligence.\
Its platform provides dashboards, automated reports, and data visualization tools.\
Northstar Analytics has a small engineering team and primarily serves medium-sized businesses.\
The company is currently focused on improving the reliability and performance of its analytics platform.\
For customer support, users can contact the support team through the company's website. The company reviews its product roadmap on a quarterly basis."""

tenant_beta_paragraph: str = """Bluepeak Systems is a software company founded in 2021. The company produces softwares for cybersecurity.\
Its platform provides dashboards, automated reports, and data visualization tools.\
Bluepeak Systems has a big engineering team and primarily serves large corporations.\
The company is currently focused on improving the reliability and performance of its analytics platform.\
For customer support, users can contact the support team through the company's website. The company reviews its product roadmap on a quarterly basis.""" 

tenant_alpha_chunks: list[str] = [s for s in tenant_alpha_paragraph.split(".")]
tenant_beta_chunks: list[str] = [s for s in tenant_beta_paragraph.split(".")]

test_tenant_alpha_doc = Document(
    tenant_id="tenant_alpha", 
    file_name="tenant_alpha.pdf", 
    text=tenant_alpha_paragraph, 
    chunks=embedder.embed(texts=tenant_alpha_chunks)
    )

test_tenant_beta_doc = Document(
    tenant_id="tenant_beta", 
    file_name="tenant_beta.pdf", 
    text=tenant_beta_paragraph, 
    chunks=embedder.embed(texts=tenant_beta_chunks)
    )

repository.add([test_tenant_alpha_doc, test_tenant_beta_doc])

# Sanity Check
repository.collection.count()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

16

In [2]:
# Query
questions = ["What is the company name?", "What year is the company founded?", "What kind of software does the company produce?"]
expected_answers_alpha = ["Northstar Analytics", "2018", "business intelligence"]
expected_answers_beta = ["Bluepeak Systems", "2021", "cybersecurity"]

alpha_contexts = []
beta_contexts = []

for question in questions:
    q_and_c_alpha = repository.query(question, "tenant_alpha", top_n=3)
    q_and_c_beta = repository.query(question, "tenant_beta", top_n=3)

    alpha_contexts.append(" ".join(q_and_c_alpha.context))
    beta_contexts.append(" ".join(q_and_c_beta.context))

for a, c in zip(expected_answers_alpha, alpha_contexts):
    print(f"Expected answer in context for tenant alpha: ", a in c)

for a, c in zip(expected_answers_beta, beta_contexts):
    print(f"Expected answer in context for tenant beta: ", a in c)

Expected answer in context for tenant alpha:  True
Expected answer in context for tenant alpha:  True
Expected answer in context for tenant alpha:  True
Expected answer in context for tenant beta:  True
Expected answer in context for tenant beta:  True
Expected answer in context for tenant beta:  True


In [4]:
tricky_question_for_alpha = "What is founding year of Bluepeak Systems?"
tricky_question_for_beta = "What is founding year of Northstar Analytics?"

q_and_c_alpha = repository.query(tricky_question_for_alpha, "tenant_alpha", top_n=3)
q_and_c_beta = repository.query(tricky_question_for_beta, "tenant_beta", top_n=3)

print(f"Context for tenant alpha: ", q_and_c_alpha.context)
print(f"Context for tenant beta: ", q_and_c_beta.context)

Context for tenant alpha:  ['Its platform provides dashboards, automated reports, and data visualization tools', 'Northstar Analytics is a software company founded in 2018', ' The company produces softwares for business intelligence']
Context for tenant beta:  ['The company is currently focused on improving the reliability and performance of its analytics platform', 'Its platform provides dashboards, automated reports, and data visualization tools', ' The company reviews its product roadmap on a quarterly basis']


As we can see, the context for tenant alpha does not contain any information about Bluepeak Systems, and the context for tenant beta does not contain any information about Northstar Analytics. 

These two tests demonstrate that the RAG system is tenant isolated. 